# Rolling-key encrypted and authenticated boot; read-only eFUSE architecture

This uses signed/encrypted variants of the **repository-generated IPI design**. No eFUSE writer or persistent enrollment is run. Timing here is host image verification and whole hardware-session wall time; **no BootROM authentication/decryption latency counter has been instrumented**. Do not compare session wall time to runtime crypto latency.

Set BUILD_BOOT=True to package fresh images from the configured runtime build with fresh HSS signing state. Never copy, rewind or reuse private signing state. Set LIVE=True to program the selected image on the exact configured SCU35 and verify programming/recovery status. Fresh hardware acceptance alone is not proof of owner-pinned trust policy; unsigned images were also accepted in earlier tests.

For encrypted combo/rolling images, the diagnostic root is public all-zero, selected from the existing eFUSE key source without writing it. Acceptance depends on existing device provisioning. This proves no secrecy and does not read out or establish default AES key values.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib
import pandas as pd
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in (base, base/'scu35') if (p/'lab.py').is_file()), None)
assert ROOT is not None, 'Open the notebook from the repository checkout'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'host'))
from lab import config, tool
from notebook import runtime, show_runtime, LiveRows, command, hardware, check_build
c = config()
OUT = Path(c['build_dir']).resolve()
print('Python:', sys.executable)
print('Build:', OUT)


In [ ]:
BUILD_BOOT = False
if BUILD_BOOT:
    check_build(c)
    command('boot', c=c)  # H15 default; may take time; creates a NEW signing-state directory
BOOT_DIR = None  # Set to the fresh <build_dir>/boot_TIMESTAMP directory
if BUILD_BOOT:
    BOOT_DIR = max(OUT.glob('boot_*'), key=lambda p:p.stat().st_mtime)
BUNDLE = Path(BOOT_DIR) if BOOT_DIR else ROOT/'evidence/boot/bundle'
print('Fresh build bundle:' if BOOT_DIR else 'Recorded public bundle:', BUNDLE)


In [ ]:
sys.path.insert(0, str(ROOT/'scripts'))
from pdi import verify_signed
PDI = (Path(BOOT_DIR)/'combo.pdi') if BOOT_DIR else ROOT/'reference/boot/combo.pdi'
verification_times = []
for _ in range(3):
    started = time.perf_counter_ns()
    verification, pieces = verify_signed(PDI.read_bytes(), BUNDLE)
    verification_times.append((time.perf_counter_ns()-started)/1e6)
assert all(verification[k] for k in ['spk_signature_valid','hash_block_signature_valid','boot_header_hash_valid'])
display(pd.DataFrame([{'method':'Python full signed-PDI verification (including file read)',
                        'run':i+1,'host_ms':v} for i,v in enumerate(verification_times)]))
display(pd.DataFrame([verification]))


In [ ]:
from pdi import header, parts, decrypt_chain
parameters = json.loads((Path(BOOT_DIR)/'public_demo_parameters.json').read_text()) if BOOT_DIR and (Path(BOOT_DIR)/'public_demo_parameters.json').exists() else None
if BOOT_DIR:
    # Fresh build_boot stores the initial PLM IV in plm.nky (local private/demo material).
    from pdi import root_material
    key, iv = root_material(Path(BOOT_DIR)/'plm.nky')
else:
    parameters = json.loads((ROOT/'reference/boot/public_demo_parameters.json').read_text())
    key = bytes.fromhex(parameters['root_key_hex']); iv = bytes.fromhex(parameters['ivs']['combo_plm'])
assert key == bytes(32), 'Only this public-zero educational flow is covered here'
h = header(PDI.read_bytes())
_, (_,_,_,_, a,b) = verify_signed(PDI.read_bytes(), BUNDLE)
started = time.perf_counter_ns()
plm_clear, plm_chain = decrypt_chain(a,key,iv)
pl_clear, pl_chain = decrypt_chain(b,key,iv[:-1]+bytes([(iv[-1]+1)&255]))
elapsed_ms = (time.perf_counter_ns()-started)/1e6
plain_path = Path(BOOT_DIR)/'plain.pdi' if BOOT_DIR else ROOT/'reference/boot/plain.pdi'
plain = plain_path.read_bytes(); ph = header(plain)
plain_a,plain_b,_,_ = parts(plain,ph,plain[0x340:0x3d0])
assert plm_clear == plain_a and pl_clear == plain_b
display(pd.DataFrame([{'operation':'Python rolling-chain decrypt + tag checks','host_ms':elapsed_ms,
                        'PLM blocks':len(plm_chain),'PL blocks':len(pl_chain),'plaintext_match':True}]))
links = json.loads((ROOT/'docs/presenter/register_links.json').read_text())
display(links['subsections']['PMCL_EFUSE'])
print('eFUSE subsection documentation only: no register access or fuse write.')


In [ ]:
LIVE = False  # Explicitly program the selected image and then recover
if LIVE:
    assert BOOT_DIR is not None, 'Choose variants of your freshly generated repository IPI build'
    session, session_wall_s = hardware(PDI, mode='boot', c=c)
    display(pd.DataFrame([{'session':str(session),'program_capture_recovery_wall_s':session_wall_s,
                           'on_chip_boot_crypto_us':None,
                           'scope':'Whole host session; not instrumented boot crypto latency'}]))
else:
    print('No hardware accessed. Python verification above was freshly timed.')
